In [31]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [32]:
df = pd.read_csv("/content/drive/MyDrive/Machine Learning/Supervised Learning/product_recemedation/amazon_products_cleaned.csv")

In [33]:
df.head()

,Unnamed: 0,Product Category,Product Subcategory,Product Name,Product URL,Product Price,Product Rating,Total Ratings,Product Image
0,0,Electronics,Smartphones,"SAMSUNG Galaxy S25 Ultra Cell Phone, 512GB AI ...",https://www.amazon.com/SAMSUNG-Smartphone-Unlo...,"$1,219.99",4.3,247,https://m.media-amazon.com/images/I/61n0lmxP5-...
1,1,Electronics,Smartphones,"SAMSUNG Galaxy A16 5G A Series Cell Phone, Unl...",https://www.amazon.com/SAMSUNG-Unlocked-Smartp...,$174.99,4.2,166,https://m.media-amazon.com/images/I/71KGkQ+KOK...
2,2,Electronics,Smartphones,Tracfone | Motorola Moto g Play 2024 | Locked ...,https://www.amazon.com/Tracfone-Motorola-5000m...,$39.88,4.3,428,https://m.media-amazon.com/images/I/71CxUvG46r...
3,3,Electronics,Smartphones,"SAMSUNG Galaxy S24 FE AI Phone, 128GB Unlocked...",https://www.amazon.com/SAMSUNG-Unlocked-Smartp...,$498.95,4.5,567,https://m.media-amazon.com/images/I/61uakkLoHx...
4,4,Electronics,Smartphones,Moto G 5G | 2024 | Unlocked | Made for US 4/12...,https://www.amazon.com/Moto-Unlocked-128GB-Cam...,$149.99,4.2,"1,677",https://m.media-amazon.com/images/I/71Sbx9zQN3...


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42433 entries, 0 to 42432
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Unnamed: 0           42433 non-null  int64  
 1   Product Category     42432 non-null  object 
 2   Product Subcategory  42432 non-null  object 
 3   Product Name         42430 non-null  object 
 4   Product URL          42432 non-null  object 
 5   Product Price        42432 non-null  object 
 6   Product Rating       42432 non-null  float64
 7   Total Ratings        42430 non-null  object 
 8   Product Image        42432 non-null  object 
dtypes: float64(1), int64(1), object(7)
memory usage: 2.9+ MB


In [35]:
print(df.shape)
print(df.columns)
# print(df.head())
print(df.isnull().sum())

(42433, 9)
Index(['Unnamed: 0', 'Product Category', 'Product Subcategory', 'Product Name',
       'Product URL', 'Product Price', 'Product Rating', 'Total Ratings',
       'Product Image'],
      dtype='object')
Unnamed: 0             0
Product Category       1
Product Subcategory    1
Product Name           3
Product URL            1
Product Price          1
Product Rating         1
Total Ratings          3
Product Image          1
dtype: int64


In [36]:
df = df.drop(columns=["Product Image","Product URL"])

In [37]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [38]:
print("Duplicate product names:", df["Product Name"].duplicated().sum())

Duplicate product names: 10071


In [39]:
print(df[df["Product Name"].isna()])
print(df[df["Product Category"].isna()])
print(df[df["Product Subcategory"].isna()])

       Unnamed: 0    Product Category Product Subcategory Product Name  \
16409       16790                 NaN                 NaN          NaN   
30950       31889  Travel Accessories   Compression socks          NaN   
30951       31890  Travel Accessories   Compression socks          NaN   

      Product Price  Product Rating Total Ratings  
16409           NaN             NaN           NaN  
30950            $.             4.3           NaN  
30951            $.             4.8           NaN  
       Unnamed: 0 Product Category Product Subcategory Product Name  \
16409       16790              NaN                 NaN          NaN   

      Product Price  Product Rating Total Ratings  
16409           NaN             NaN           NaN  
       Unnamed: 0 Product Category Product Subcategory Product Name  \
16409       16790              NaN                 NaN          NaN   

      Product Price  Product Rating Total Ratings  
16409           NaN             NaN           NaN  


In [40]:
df = df.rename(columns={"Unnamed: 0": "product_id"})

In [41]:
text_columns = [
    "Product Category",
    "Product Subcategory",
    "Product Name"
]

df[text_columns] = df[text_columns].fillna("")

df["combined_text"] = (
    df["Product Name"] + " " +
    df["Product Category"] + " " +
    df["Product Subcategory"]
)

In [44]:
print(df[
    [
        "product_id",
        "Product Name",
        "Product Category",
        "Product Subcategory",
        "combined_text"
    ]
].head(10))

   product_id                                       Product Name  \
0           0  SAMSUNG Galaxy S25 Ultra Cell Phone, 512GB AI ...   
1           1  SAMSUNG Galaxy A16 5G A Series Cell Phone, Unl...   
2           2  Tracfone | Motorola Moto g Play 2024 | Locked ...   
3           3  SAMSUNG Galaxy S24 FE AI Phone, 128GB Unlocked...   
4           4  Moto G 5G | 2024 | Unlocked | Made for US 4/12...   
5           5  Moto G Play | 2024 | Unlocked | Made for US 4/...   
6           6  Samsung Galaxy A15 (SM-155M/DSN), 128GB 6GB RA...   
7           7  Google Pixel Fold - Unlocked Android 5G Smartp...   
8           8  Bold K50 | 5G| 2024 | 3-Day Battery | Unlocked...   
9           9  SAMSUNG Galaxy A35 5G A Series Cell Phone, 128...   

  Product Category Product Subcategory  \
0      Electronics         Smartphones   
1      Electronics         Smartphones   
2      Electronics         Smartphones   
3      Electronics         Smartphones   
4      Electronics         Smartphones   

In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english"
)

tfidf_matrix = tfidf.fit_transform(df["combined_text"])

In [46]:
print(tfidf_matrix.shape)

(42433, 34184)


In [47]:
print("Products:", tfidf_matrix.shape[0])
print("Features:", tfidf_matrix.shape[1])

Products: 42433
Features: 34184


In [49]:
from sklearn.metrics.pairwise import cosine_similarity

product_index = 0

similarity_scores = cosine_similarity(
    tfidf_matrix[product_index],
    tfidf_matrix
)

In [50]:
print(similarity_scores.shape)

(1, 42433)


In [51]:
print(similarity_scores)

[[1.         0.42162589 0.11334663 ... 0.         0.         0.01521613]]


In [52]:
similarity_scores = similarity_scores.flatten()

similar_indices = similarity_scores.argsort()[::-1]

print(similar_indices[:10])

[ 0 28 12  3 30 14  1 11  9 17]


In [53]:
from sklearn.metrics.pairwise import cosine_similarity

def recommend_products(product_id, n=5):
    # Find the row/index of the product
    product_index = df.index[df["product_id"] == product_id][0]

    # Calculate similarity with all products
    similarity_scores = cosine_similarity(
        tfidf_matrix[product_index],
        tfidf_matrix
    ).flatten()

    # Sort from highest similarity to lowest
    similar_indices = similarity_scores.argsort()[::-1]

    # Remove the product itself
    similar_indices = similar_indices[similar_indices != product_index]

    # Get top N products
    top_indices = similar_indices[:n]

    # Return product information
    recommendations = df.iloc[top_indices][
        [
            "product_id",
            "Product Name",
            "Product Category",
            "Product Subcategory",
            "Product Price",
            "Product Rating"
        ]
    ].copy()

    return recommendations

In [54]:
recommendations = recommend_products(0, 5)

print(recommendations)

    product_id                                       Product Name  \
28          28  SAMSUNG Galaxy Z Fold 6 AI Cell Phone, 512GB U...   
12          12  Samsung Galaxy S24 Ultra Cell Phone, 512GB AI ...   
3            3  SAMSUNG Galaxy S24 FE AI Phone, 128GB Unlocked...   
30          30  Samsung Galaxy S22 Smartphone, Factory Unlocke...   
14          14  Samsung Galaxy A03s Cell Phone, AT&T GSM Unloc...   

   Product Category Product Subcategory Product Price  Product Rating  
28      Electronics         Smartphones    $1,549.97              4.3  
12      Electronics         Smartphones    $1,078.99              4.5  
3       Electronics         Smartphones      $498.95              4.5  
30      Electronics         Smartphones      $258.44              4.0  
14      Electronics         Smartphones       $69.49              3.8  
